In [ ]:
from __future__ import annotations

import json
import pickle
import random
import re
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import networkx as nx

import nltk

from scipy.sparse import csr_matrix, hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score


SEED = 40
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path("corpus")
OUT_DIR = Path("outputs")
STACK_OUT_DIR = Path("outputs_stacking")
STACK_OUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    stop_words = nltk.corpus.stopwords.words("portuguese")
except LookupError:
    nltk.download("stopwords", quiet=True)
    stop_words = nltk.corpus.stopwords.words("portuguese")

In [ ]:
def load_jsonl_df(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"JSON inválido em {path} na linha {i}: {e}") from e
    if not rows:
        raise ValueError(f"Arquivo vazio: {path}")
    return pd.DataFrame(rows)

def create_x_y(df: pd.DataFrame) -> Tuple[pd.Series, pd.Series]:
    if "text" not in df.columns or "label" not in df.columns:
        raise ValueError(f"Esperado 'text' e 'label'. Encontrei: {list(df.columns)}")
    x = df["text"].astype(str)
    y = df["label"].astype(int).values
    return x, y

df_train = load_jsonl_df(DATA_DIR / "train.jsonl")
df_test  = load_jsonl_df(DATA_DIR / "test.jsonl")

x_train, y_train = create_x_y(df_train)
x_test, y_test   = create_x_y(df_test)

In [ ]:
def load_graph_pkl(path: Path) -> nx.Graph:
    with path.open("rb") as f:
        obj = pickle.load(f)
    if not isinstance(obj, nx.Graph):
        raise TypeError(f"{path.name} não é nx.Graph (tipo={type(obj)})")
    return obj

G_cooc = load_graph_pkl(OUT_DIR / "KG_cooc.gpickle")
G_ppmi = load_graph_pkl(OUT_DIR / "KG_ppmi.gpickle")
G_pun  = load_graph_pkl(OUT_DIR / "KG_puncontext.gpickle")

graphs: Dict[str, nx.Graph] = {
    "cooc": G_cooc,
    "ppmi": G_ppmi,
    "pun": G_pun,
}

In [ ]:
from scipy.sparse import diags, identity

GRAPH_EMB_DIM = 128
GRAPH_EXTRA = 2
GRAPH_FEATURE_DIM = GRAPH_EMB_DIM + GRAPH_EXTRA

TOKEN_RE = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ]+", re.UNICODE)

def tokenize_for_graph(text: str) -> List[str]:
    return [t.lower() for t in TOKEN_RE.findall(text)]

def nx_to_sparse_adjacency(G: nx.Graph, nodelist: List[str], weight: str = "weight") -> csr_matrix:
    if hasattr(nx, "to_scipy_sparse_array"):
        A = nx.to_scipy_sparse_array(G, nodelist=nodelist, weight=weight, dtype=float, format="csr")
        return csr_matrix(A)
    return nx.to_scipy_sparse_matrix(G, nodelist=nodelist, weight=weight, dtype=float, format="csr")

def normalize_adjacency(A: csr_matrix, add_self_loops: bool = True) -> csr_matrix:
    A = A.tocsr(copy=True)
    if add_self_loops:
        A = A + identity(A.shape[0], format="csr", dtype=A.dtype)

    deg = np.array(A.sum(axis=1)).ravel()
    deg[deg == 0] = 1.0
    inv_sqrt = 1.0 / np.sqrt(deg)

    D_inv_sqrt = diags(inv_sqrt, format="csr")
    return (D_inv_sqrt @ A @ D_inv_sqrt).tocsr()

def build_node_embeddings_svd(G: nx.Graph, dim: int, seed: int) -> Tuple[Dict[str, int], np.ndarray]:
    nodelist = list(G.nodes())
    node_to_idx = {n: i for i, n in enumerate(nodelist)}
    A = nx_to_sparse_adjacency(G, nodelist=nodelist, weight="weight")

    if A.nnz > 0:
        A = A.tocsr(copy=True)
        A.data = np.log1p(A.data)

    A_norm = normalize_adjacency(A, add_self_loops=True)

    svd = TruncatedSVD(n_components=dim, random_state=seed)
    node_emb = svd.fit_transform(A_norm).astype(np.float32)
    return node_to_idx, node_emb


def doc_features_from_graph(texts: List[str], node_to_idx: Dict[str, int], node_emb: np.ndarray) -> np.ndarray:
    n = len(texts)
    feats = np.zeros((n, GRAPH_FEATURE_DIM), dtype=np.float32)
    for i, txt in enumerate(texts):
        toks = tokenize_for_graph(txt)
        if not toks:
            continue
        idxs = [node_to_idx[t] for t in toks if t in node_to_idx]
        hits = len(idxs)
        total = len(toks)
        if hits > 0:
            feats[i, :GRAPH_EMB_DIM] = node_emb[idxs].mean(axis=0)
        feats[i, GRAPH_EMB_DIM] = (hits / total) if total > 0 else 0.0
        feats[i, GRAPH_EMB_DIM + 1] = np.log1p(hits)
    return feats

node_models: Dict[str, Dict[str, object]] = {}
for k, G in graphs.items():
    node_to_idx, node_emb = build_node_embeddings_svd(G, dim=GRAPH_EMB_DIM, seed=SEED)
    node_models[k] = {"node_to_idx": node_to_idx, "node_emb": node_emb}

graph_feats: Dict[str, Dict[str, np.ndarray]] = {k: {} for k in graphs.keys()}
for k in graphs.keys():
    node_to_idx = node_models[k]["node_to_idx"]
    node_emb = node_models[k]["node_emb"]
    graph_feats[k]["train"] = doc_features_from_graph(x_train.tolist(), node_to_idx, node_emb)
    graph_feats[k]["test"]  = doc_features_from_graph(x_test.tolist(),  node_to_idx, node_emb)

def stack_graph_blocks(keys: List[str], split: str) -> np.ndarray:
    blocks = [graph_feats[k][split] for k in keys]
    return np.hstack(blocks).astype(np.float32)

In [ ]:
def make_vectorizer() -> TfidfVectorizer:
    return TfidfVectorizer(ngram_range=(1, 2), stop_words=stop_words)

def make_baseline_voter() -> VotingClassifier:
    rf_model = RandomForestClassifier(n_estimators=100, criterion="entropy", max_depth=15, random_state=SEED)
    lr_model = LogisticRegression(random_state=SEED, max_iter=2000)
    svm_model = SVC(probability=True, random_state=SEED)
    return VotingClassifier(
        estimators=[("rf", rf_model), ("lr", lr_model), ("svm", svm_model)],
        voting="soft",
        n_jobs=1,
    )

def build_fused_mats(vec: TfidfVectorizer, Xtr_text, Xsp_text, Xtr_g_raw: np.ndarray, Xsp_g_raw: np.ndarray):
    scaler = StandardScaler()
    scaler.fit(Xtr_g_raw)
    Xtr_g = scaler.transform(Xtr_g_raw).astype(np.float32)
    Xsp_g = scaler.transform(Xsp_g_raw).astype(np.float32)

    Xtr = hstack([Xtr_text, csr_matrix(Xtr_g)], format="csr")
    Xsp = hstack([Xsp_text, csr_matrix(Xsp_g)], format="csr")

    Xtr = ensure_csr_writeable(Xtr)
    Xsp = ensure_csr_writeable(Xsp)

    return Xtr, Xsp

In [ ]:
from scipy.sparse import isspmatrix_csr

def ensure_csr_writeable(X):
    X = X.tocsr(copy=True)
    X.sort_indices()
    if hasattr(X.data, "flags") and not X.data.flags.writeable:
        X.data = np.array(X.data, copy=True)
    if hasattr(X.indices, "flags") and not X.indices.flags.writeable:
        X.indices = np.array(X.indices, copy=True)
    if hasattr(X.indptr, "flags") and not X.indptr.flags.writeable:
        X.indptr = np.array(X.indptr, copy=True)
    return X

In [ ]:
def proba_to_meta_features(proba: np.ndarray) -> np.ndarray:
    p = np.clip(proba.astype(np.float32), 1e-7, 1.0 - 1e-7)
    maxp = np.max(p, axis=1, keepdims=True)
    ent = -(p * np.log(p)).sum(axis=1, keepdims=True)
    return np.hstack([p, maxp, ent]).astype(np.float32)

In [ ]:
candidate_specs = [
    ("text", []),
    ("cooc", ["cooc"]),
    ("ppmi", ["ppmi"]),
    ("pun",  ["pun"]),
    ("all",  ["cooc", "ppmi", "pun"]),
]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

oof_full = np.zeros((len(x_train), 4 * len(candidate_specs)), dtype=np.float32)

x_train_list = x_train.tolist()

for fold_id, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_train)), y_train), start=1):
    x_tr = [x_train_list[i] for i in tr_idx]
    x_va = [x_train_list[i] for i in va_idx]
    y_tr = y_train[tr_idx]

    vec = make_vectorizer()
    Xtr_text = vec.fit_transform(x_tr)
    Xva_text = vec.transform(x_va)

    for m_i, (name, gkeys) in enumerate(candidate_specs):
        model = make_baseline_voter()

        if name == "text":
            model.fit(Xtr_text, y_tr)
            proba = model.predict_proba(Xva_text)
        else:
            Xtr_g_raw = stack_graph_blocks(gkeys, "train")[tr_idx]
            Xva_g_raw = stack_graph_blocks(gkeys, "train")[va_idx]
            Xtr_fused, Xva_fused = build_fused_mats(vec, Xtr_text, Xva_text, Xtr_g_raw, Xva_g_raw)
            model.fit(Xtr_fused, y_tr)
            proba = model.predict_proba(Xva_fused)

        feats = proba_to_meta_features(proba)
        oof_full[va_idx, (4*m_i):(4*m_i+4)] = feats

In [ ]:
def oof_acc_for_spec(oof_mat: np.ndarray, spec_index: int) -> float:
    p = oof_mat[:, (4*spec_index):(4*spec_index+2)]
    y_pred = np.argmax(p, axis=1)
    return float((y_pred == y_train).mean())

accs = [(name, oof_acc_for_spec(oof_full, i)) for i, (name, _) in enumerate(candidate_specs)]
accs_sorted = sorted(accs, key=lambda t: t[1], reverse=True)

selected_specs_all  = candidate_specs[:]

sel_idx_all  = list(range(len(candidate_specs)))

oof_all  = np.hstack([oof_full[:, (4*i):(4*i+4)] for i in sel_idx_all ]).astype(np.float32)

print("OOF accs:", accs_sorted)
print("Specs ALL :", [n for n, _ in selected_specs_all])
print("oof_all  shape:", oof_all.shape)

OOF accs: [('text', 0.7453634085213032), ('ppmi', 0.5428571428571428), ('cooc', 0.5343358395989974), ('all', 0.4395989974937343), ('pun', 0.3493734335839599)]
Specs ALL : ['text', 'cooc', 'ppmi', 'pun', 'all']
oof_all  shape: (3990, 20)


In [ ]:
meta_cooc = RandomForestClassifier(n_estimators=100, criterion="entropy", max_depth=15, random_state=SEED)
meta_ppmi = RandomForestClassifier(n_estimators=100, criterion="entropy", max_depth=15, random_state=SEED)
meta_pun  = RandomForestClassifier(n_estimators=100, criterion="entropy", max_depth=15, random_state=SEED)
meta_all  = RandomForestClassifier(n_estimators=100, criterion="entropy", max_depth=15, random_state=SEED)

idx = {n: i for i, (n, _) in enumerate(candidate_specs)}

oof_text_cooc = np.hstack([oof_full[:, (4*idx["text"]):(4*idx["text"]+4)],
                           oof_full[:, (4*idx["cooc"]):(4*idx["cooc"]+4)]]).astype(np.float32)

oof_text_ppmi = np.hstack([oof_full[:, (4*idx["text"]):(4*idx["text"]+4)],
                           oof_full[:, (4*idx["ppmi"]):(4*idx["ppmi"]+4)]]).astype(np.float32)

oof_text_pun  = np.hstack([oof_full[:, (4*idx["text"]):(4*idx["text"]+4)],
                           oof_full[:, (4*idx["pun"]):(4*idx["pun"]+4)]]).astype(np.float32)

meta_cooc.fit(oof_text_cooc, y_train)
meta_ppmi.fit(oof_text_ppmi, y_train)
meta_pun.fit(oof_text_pun, y_train)

meta_all.fit(oof_all, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'entropy'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metri

In [ ]:
vec_full = make_vectorizer()
Xtr_text_full = ensure_csr_writeable(vec_full.fit_transform(x_train.tolist()))
Xte_text_full = ensure_csr_writeable(vec_full.transform(x_test.tolist()))

probas_test_by_name = {}

Z_test_all = np.zeros((len(x_test), 4 * len(candidate_specs)), dtype=np.float32)

for m_i, (name, gkeys) in enumerate(candidate_specs):
    model = make_baseline_voter()

    if name == "text":
        model.fit(Xtr_text_full, y_train)
        proba_test = model.predict_proba(Xte_text_full)
    else:
        Xtr_g_raw = stack_graph_blocks(gkeys, "train")
        Xte_g_raw = stack_graph_blocks(gkeys, "test")
        Xtr_fused, Xte_fused = build_fused_mats(vec_full, Xtr_text_full, Xte_text_full, Xtr_g_raw, Xte_g_raw)
        model.fit(Xtr_fused, y_train)
        proba_test = model.predict_proba(Xte_fused)

    probas_test_by_name[name] = proba_test.astype(np.float32)
    Z_test_all[:, (4*m_i):(4*m_i+4)] = proba_to_meta_features(proba_test)

idx = {n: i for i, (n, _) in enumerate(candidate_specs)}

Z_test_text_cooc = np.hstack([Z_test_all[:, (4*idx["text"]):(4*idx["text"]+4)],
                              Z_test_all[:, (4*idx["cooc"]):(4*idx["cooc"]+4)]]).astype(np.float32)

Z_test_text_ppmi = np.hstack([Z_test_all[:, (4*idx["text"]):(4*idx["text"]+4)],
                              Z_test_all[:, (4*idx["ppmi"]):(4*idx["ppmi"]+4)]]).astype(np.float32)

Z_test_text_pun  = np.hstack([Z_test_all[:, (4*idx["text"]):(4*idx["text"]+4)],
                              Z_test_all[:, (4*idx["pun"]):(4*idx["pun"]+4)]]).astype(np.float32)

In [ ]:
def predict_from_proba(proba: np.ndarray) -> np.ndarray:
    return np.argmax(proba, axis=1)

rows = []
for name, _ in candidate_specs:
    proba = probas_test_by_name[name]
    y_pred = predict_from_proba(proba)
    acc = accuracy_score(y_test, y_pred)
    print(f"\n=== {name.upper()} (TEST) ===")
    print("acc:", acc)
    print(classification_report(y_test, y_pred))
    rows.append((name, acc))

print("\nResumo (acc):")
for name, acc in sorted(rows, key=lambda x: x[1], reverse=True):
    print(f"{name:>6} -> {acc:.4f}")


=== TEXT (TEST) ===
acc: 0.7973684210526316
              precision    recall  f1-score   support

           0       0.79      0.81      0.80       570
           1       0.80      0.79      0.80       570

    accuracy                           0.80      1140
   macro avg       0.80      0.80      0.80      1140
weighted avg       0.80      0.80      0.80      1140


=== COOC (TEST) ===
acc: 0.5894736842105263
              precision    recall  f1-score   support

           0       0.59      0.61      0.60       570
           1       0.59      0.56      0.58       570

    accuracy                           0.59      1140
   macro avg       0.59      0.59      0.59      1140
weighted avg       0.59      0.59      0.59      1140


=== PPMI (TEST) ===
acc: 0.5833333333333334
              precision    recall  f1-score   support

           0       0.58      0.60      0.59       570
           1       0.59      0.57      0.58       570

    accuracy                           0.58    

In [ ]:
y_pred_stack_cooc = meta_cooc.predict(Z_test_text_cooc)
acc_stack_cooc = accuracy_score(y_test, y_pred_stack_cooc)
print("STACKING TEXT+COOC (A+B) TEST ACC:", acc_stack_cooc)
print(classification_report(y_test, y_pred_stack_cooc))

y_pred_stack_ppmi = meta_ppmi.predict(Z_test_text_ppmi)
acc_stack_ppmi = accuracy_score(y_test, y_pred_stack_ppmi)
print("STACKING TEXT+PPMI (A+B) TEST ACC:", acc_stack_ppmi)
print(classification_report(y_test, y_pred_stack_ppmi))

y_pred_stack_pun = meta_pun.predict(Z_test_text_pun)
acc_stack_pun = accuracy_score(y_test, y_pred_stack_pun)
print("STACKING TEXT+PUN (A+B) TEST ACC:", acc_stack_pun)
print(classification_report(y_test, y_pred_stack_pun))

y_pred_stack_all = meta_all.predict(Z_test_all)
acc_stack_all = accuracy_score(y_test, y_pred_stack_all)
print("STACKING ALL (A+B) TEST ACC:", acc_stack_all)
print(classification_report(y_test, y_pred_stack_all))

baseline_vec = make_vectorizer()
Xtr_base = ensure_csr_writeable(baseline_vec.fit_transform(x_train.tolist()))
Xte_base = ensure_csr_writeable(baseline_vec.transform(x_test.tolist()))

baseline_model = make_baseline_voter()
baseline_model.fit(Xtr_base, y_train)

y_pred_base = baseline_model.predict(Xte_base)
acc_base = accuracy_score(y_test, y_pred_base)
print("BASELINE TEST ACC:", acc_base)
print(classification_report(y_test, y_pred_base))

y_pred_text_cooc = np.argmax(probas_test_by_name["cooc"], axis=1)
y_pred_text_ppmi = np.argmax(probas_test_by_name["ppmi"], axis=1)
y_pred_text_pun  = np.argmax(probas_test_by_name["pun"], axis=1)
y_pred_text_all  = np.argmax(probas_test_by_name["all"], axis=1)

STACKING TEXT+COOC (A+B) TEST ACC: 0.8026315789473685
              precision    recall  f1-score   support

           0       0.79      0.82      0.81       570
           1       0.81      0.79      0.80       570

    accuracy                           0.80      1140
   macro avg       0.80      0.80      0.80      1140
weighted avg       0.80      0.80      0.80      1140

STACKING TEXT+PPMI (A+B) TEST ACC: 0.7859649122807018
              precision    recall  f1-score   support

           0       0.78      0.80      0.79       570
           1       0.80      0.77      0.78       570

    accuracy                           0.79      1140
   macro avg       0.79      0.79      0.79      1140
weighted avg       0.79      0.79      0.79      1140

STACKING TEXT+PUN (A+B) TEST ACC: 0.8
              precision    recall  f1-score   support

           0       0.80      0.81      0.80       570
           1       0.80      0.79      0.80       570

    accuracy                        

In [ ]:
PUNS_PATH = Path("corpus/puns.json")

with PUNS_PATH.open("r", encoding="utf-8") as f:
    puns_meta = json.load(f)

def classify_pun_type(signs):
    if not signs:
        return "nenhum"

    has_homograph = any(bool(s.get("homograph", False)) for s in signs)
    has_homophone = any(bool(s.get("homophone", False)) for s in signs)
    has_both = any(
        bool(s.get("homograph", False)) and bool(s.get("homophone", False))
        for s in signs
    )

    if has_both:
        return "ambos"
    if has_homograph:
        return "homografos"
    if has_homophone:
        return "homonimos"
    return "nenhum"

puns_df = pd.DataFrame(puns_meta)
puns_df["pun_type"] = puns_df["signs"].apply(classify_pun_type)
puns_df = puns_df[["text", "pun_type", "signs"]].drop_duplicates(subset=["text"])

analysis_df = df_test.copy().reset_index(drop=True)
analysis_df["y_true"] = y_test
analysis_df = analysis_df.merge(puns_df, on="text", how="left")
analysis_df["pun_type"] = analysis_df["pun_type"].fillna("nenhum")
analysis_df["signs"] = analysis_df["signs"].apply(
    lambda x: x if isinstance(x, list) else []
)

# ANALISAR APENAS O MODELO FINAL COMPLETO
models_to_analyze = {
    "stack_cooc": y_pred_stack_cooc,
    "stack_ppmi": y_pred_stack_ppmi,
    "stack_pun": y_pred_stack_pun,
    "stack_all": y_pred_stack_all,
}

def compute_confusion_counts(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    return {"TP": tp, "TN": tn, "FP": fp, "FN": fn}

def add_confusion_cell(df):
    df = df.copy()
    df["confusion_cell"] = np.where(
        (df["y_true"] == 1) & (df["y_pred"] == 1), "TP",
        np.where(
            (df["y_true"] == 0) & (df["y_pred"] == 0), "TN",
            np.where(
                (df["y_true"] == 0) & (df["y_pred"] == 1), "FP",
                "FN"
            )
        )
    )
    return df

def summarize_pun_types(df):
    order = ["homonimos", "homografos", "ambos", "nenhum"]
    counts = df["pun_type"].value_counts()
    return {ptype: int(counts.get(ptype, 0)) for ptype in order}

for model_name, y_pred in models_to_analyze.items():
    print(f"\n{'='*60}")
    print(f"MODELO FINAL: {model_name}")
    print(f"{'='*60}")

    counts = compute_confusion_counts(analysis_df["y_true"].values, y_pred)
    print("TP, TN, FP, FN:")
    print(counts)

    tmp = analysis_df.copy()
    tmp["y_pred"] = y_pred
    tmp = add_confusion_cell(tmp)

    print("\nDistribuição por tipo de trocadilho nos FN:")
    print(summarize_pun_types(tmp[tmp["confusion_cell"] == "FN"]))

    print("\nDistribuição por tipo de trocadilho nos TP:")
    print(summarize_pun_types(tmp[tmp["confusion_cell"] == "TP"]))

    print("\nDistribuição por tipo de trocadilho nos FP:")
    print(summarize_pun_types(tmp[tmp["confusion_cell"] == "FP"]))

    print("\nDistribuição por tipo de trocadilho nos TN:")
    print(summarize_pun_types(tmp[tmp["confusion_cell"] == "TN"]))

rows = []

for model_name, y_pred in models_to_analyze.items():
    tmp = analysis_df.copy()
    tmp["y_pred"] = y_pred
    tmp = add_confusion_cell(tmp)
    tmp["model"] = model_name

    cols = ["model", "text", "label", "y_true", "y_pred", "confusion_cell", "pun_type", "signs"]
    rows.append(tmp[cols])

pun_cases_df = pd.concat(rows, ignore_index=True)
pun_cases_df.to_csv(STACK_OUT_DIR / "pun_cases.csv", index=False)

print(f"\nArquivo único salvo em: {STACK_OUT_DIR / 'pun_cases.csv'}")


MODELO FINAL: stack_cooc
TP, TN, FP, FN:
{'TP': 448, 'TN': 467, 'FP': 103, 'FN': 122}

Distribuição por tipo de trocadilho nos FN:
{'homonimos': 32, 'homografos': 1, 'ambos': 19, 'nenhum': 70}

Distribuição por tipo de trocadilho nos TP:
{'homonimos': 82, 'homografos': 0, 'ambos': 80, 'nenhum': 286}

Distribuição por tipo de trocadilho nos FP:
{'homonimos': 0, 'homografos': 0, 'ambos': 0, 'nenhum': 103}

Distribuição por tipo de trocadilho nos TN:
{'homonimos': 0, 'homografos': 0, 'ambos': 0, 'nenhum': 467}

MODELO FINAL: stack_ppmi
TP, TN, FP, FN:
{'TP': 439, 'TN': 457, 'FP': 113, 'FN': 131}

Distribuição por tipo de trocadilho nos FN:
{'homonimos': 30, 'homografos': 1, 'ambos': 24, 'nenhum': 76}

Distribuição por tipo de trocadilho nos TP:
{'homonimos': 84, 'homografos': 0, 'ambos': 75, 'nenhum': 280}

Distribuição por tipo de trocadilho nos FP:
{'homonimos': 0, 'homografos': 0, 'ambos': 0, 'nenhum': 113}

Distribuição por tipo de trocadilho nos TN:
{'homonimos': 0, 'homografos': 0,